In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# If you don't have shapely installed:
#   pip install shapely
from shapely.geometry import LineString
from shapely.ops import clip_by_rect

In [ ]:
# --- Plot bounds (as requested) ---
x_left, x_right = 0.1, 91.2
y_bottom, y_top = 1e-5, 1.0  # y max = 10^0

files = {
    "B": "B.csv",                 # blue
    "Charm": "charm.csv",         # (you called it yellow)
    "KpiX": "KpiX.csv",           # green
    "leP": "Lep.csv",             # red (not filled)
    "LHC": "LHC.csv",             # purple (not filled)
    "SN1987A": "SN1987A.csv",     # brown
}

def clipped_segments_xy(x, y):
    """Clip a polyline to the plot rectangle and return list of (xs, ys) segments."""
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if len(x) < 2:
        return []

    line = LineString(np.column_stack([x, y]))
    clipped = clip_by_rect(line, x_left, y_bottom, x_right, y_top)

    segs = []
    if clipped.is_empty:
        return segs

    if clipped.geom_type == "LineString":
        coords = np.array(clipped.coords)
        segs.append((coords[:, 0], coords[:, 1]))
    elif clipped.geom_type == "MultiLineString":
        for g in clipped.geoms:
            coords = np.array(g.coords)
            segs.append((coords[:, 0], coords[:, 1]))
    return segs

def touches_left(xs, tol=1e-12):
    return np.any(np.isclose(xs, x_left, atol=tol, rtol=0))

def touches_top(ys, tol=1e-12):
    return np.any(np.isclose(ys, y_top, atol=tol, rtol=0))

def add_fill(ax, xs, ys, mode, color, alpha=0.20, zorder=1):
    """
    Fill polygon region defined by curve segment (xs,ys) and plot boundaries.

    mode:
      - "left":       fill between y-axis (x_left) and the segment
      - "top":        fill between top boundary (y_top) and the segment
      - "upper_left": fill the upper-left region using both top and left boundaries
    """
    xs = np.asarray(xs); ys = np.asarray(ys)
    if len(xs) < 2:
        return

    if mode == "left":
        verts = [(x_left, ys[0])] + list(zip(xs, ys)) + [(x_left, ys[-1])]
    elif mode == "top":
        verts = [(xs[0], y_top)] + list(zip(xs, ys)) + [(xs[-1], y_top)]
    elif mode == "upper_left":
        verts = [(x_left, y_top), (x_left, ys[0])] + list(zip(xs, ys)) + [(xs[-1], y_top)]
    else:
        return

    patch = Polygon(
        verts, closed=True,
        facecolor=color, edgecolor="none",
        alpha=alpha, zorder=zorder
    )
    ax.add_patch(patch)

# --- Draw ---
fig, ax = plt.subplots()

# 1) Plot the lines FIRST using original order (do NOT sort),
#    so the curve shapes remain exactly as in your "correct lines" version.
line_color = {}
for label, path in files.items():
    arr = pd.read_csv(path, header=None).values
    x, y = arr[:, 0], arr[:, 1]
    (ln,) = ax.loglog(x, y, linewidth=1.8, label=label, zorder=3)
    line_color[label] = ln.get_color()

# 2) Add fills (only for the ones you specified: B, Charm, KpiX, SN1987A)
for label, path in files.items():
    if label not in {"B", "Charm", "KpiX", "SN1987A"}:
        continue

    arr = pd.read_csv(path, header=None).values
    x, y = arr[:, 0], arr[:, 1]

    segs = clipped_segments_xy(x, y)
    for xs, ys in segs:
        if label == "SN1987A":
            # brown: between y-axis and brown curve
            add_fill(ax, xs, ys, mode="left", color=line_color[label])

        elif label == "B":
            # blue: between top boundary and blue curve
            add_fill(ax, xs, ys, mode="top", color=line_color[label])

        elif label == "Charm":
            # yellow: curve touches y-axis and top -> fill upper-left region
            add_fill(ax, xs, ys, mode="upper_left", color=line_color[label])

        elif label == "KpiX":
            # green: has two parts; decide per clipped segment
            L = touches_left(xs)
            T = touches_top(ys)
            if L and T:
                add_fill(ax, xs, ys, mode="upper_left", color=line_color[label])
            elif L:
                add_fill(ax, xs, ys, mode="left", color=line_color[label])
            elif T:
                add_fill(ax, xs, ys, mode="top", color=line_color[label])

# --- Axes styling (as requested) ---
ax.set_xlim(x_left, x_right)
ax.set_ylim(y_bottom, y_top)
ax.set_xlabel("Mass [GeV]")
ax.set_ylabel("Coupling / Scale")
ax.legend(frameon=False)

plt.show()